<a href="https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane confirmed:** Lane 2 — Refresh / Content Opportunity Scoring. Locking this in for the rest of the track.

**Working month:** `2026-03` (same mid-panel month as ML-04, avoiding the sealed final month).


In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, pandas as pd

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

MONTH_START = "2026-03-01"
MONTH_END   = "2026-04-01"  # exclusive

print("Connected. Working month:", MONTH_START, "to", MONTH_END)


Connected. Working month: 2026-03-01 to 2026-04-01


## 1. My rule and its reason codes — checked against two real signals first

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule, in plain words:** flag a page for a title/meta rewrite when it sits in a position band where FlyRank's real CTR-fix flag would also look (positions 4–20 — visible, but not the top 3), and its actual click-through rate falls meaningfully below what pages in that same position band normally get. This only fires when the page has enough impressions to trust the number.

**Reason code:** `ctr_below_tier_expectation`
**Action label:** `review_title_meta`

Before encoding this, I'm checking the two signals it leans on — one is the exact signal behind FlyRank's real CTR-fix flag from this week's session.

---

### Signal check 1 (flag-linked): does CTR really fall as position gets worse?

This is the signal directly behind FlyRank's CTR-fix flag. If it's not true in my slice, my whole rule is built on sand.


In [2]:
signal_1 = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            AVG(gsc_avg_position)                               AS avg_position,
            SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)   AS ctr
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 500
    )
    SELECT
        CASE
            WHEN avg_position <= 3  THEN 'page_1_top3'
            WHEN avg_position <= 10 THEN 'top_10'
            WHEN avg_position <= 20 THEN 'top_20'
            ELSE 'beyond_20'
        END AS position_tier,
        COUNT(*)      AS n,
        ROUND(AVG(ctr), 4) AS mean_ctr
    FROM page_month
    GROUP BY 1
    ORDER BY MIN(avg_position)
""").df()

signal_1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_tier,n,mean_ctr
0,page_1_top3,7145,0.0038
1,top_10,31845,0.0032
2,top_20,11774,0.0026
3,beyond_20,11160,0.0013


**Verdict — Signal 1: CONFIRMED**

CTR falls cleanly and monotonically as position tier worsens: page_1_top3 = 0.38% (n=7,145), top_10 = 0.32% (n=31,845), top_20 = 0.26% (n=11,774), beyond_20 = 0.13% (n=11,160). The signal behind FlyRank's real CTR-fix flag holds up in this slice — my rule is built on solid ground.

### Signal check 2: does the 4–10 position band actually carry real search volume?

This backs the "worth fixing" half of the rule — a low CTR on a page nobody searches for isn't a quick win, it's noise. If pages in the 4–10 band have real impression volume, targeting them is a genuine opportunity, the same reasoning behind FlyRank's quick-win flag.


In [3]:
signal_2 = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            AVG(gsc_avg_position)   AS avg_position,
            SUM(gsc_impressions)    AS total_impressions
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
        GROUP BY 1
    )
    SELECT
        CASE
            WHEN avg_position <= 3  THEN 'page_1_top3'
            WHEN avg_position <= 10 THEN 'top_10'
            WHEN avg_position <= 20 THEN 'top_20'
            ELSE 'beyond_20'
        END AS position_tier,
        COUNT(*)              AS n,
        ROUND(AVG(total_impressions), 0) AS mean_impressions,
        ROUND(MEDIAN(total_impressions), 0) AS median_impressions
    FROM page_month
    GROUP BY 1
    ORDER BY MIN(avg_position)
""").df()

signal_2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_tier,n,mean_impressions,median_impressions
0,page_1_top3,17578,2237.0,128.0
1,top_10,81988,1795.0,201.0
2,top_20,32203,1082.0,257.0
3,beyond_20,199668,297.0,0.0


**Verdict — Signal 2: CONFIRMED**

The top_10 tier carries real, non-trivial volume: n=81,988 pages, median 201 impressions per page (mean 1,795 — the gap shows a right-skewed distribution, with a handful of very large pages pulling the mean up, but the median alone confirms typical pages here have genuine search demand). The "quick win" reasoning holds up in this slice — these aren't noise.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Score:** for pages in the `top_10`/`top_20` band with enough volume, `score = total_impressions × max(0, expected_ctr_for_tier − actual_ctr)` — bigger gap and bigger audience both push a page up the queue. Everything else gets excluded (no action needed) rather than scored low, to keep the top of the queue honest.


In [4]:
import os

page_month = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position)                               AS avg_position,
        SUM(gsc_impressions)                                AS total_impressions,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)   AS ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 500
""").df()

def position_tier(p):
    if p <= 3:  return "page_1_top3"
    if p <= 10: return "top_10"
    if p <= 20: return "top_20"
    return "beyond_20"

page_month["position_tier"] = page_month["avg_position"].apply(position_tier)

# expected CTR per tier, measured from this same slice (not invented)
expected_ctr = page_month.groupby("position_tier")["ctr"].mean().to_dict()
page_month["expected_ctr_tier"] = page_month["position_tier"].map(expected_ctr)

# only the tiers my rule targets
eligible = page_month[page_month["position_tier"].isin(["top_10", "top_20"])].copy()

eligible["ctr_gap"] = (eligible["expected_ctr_tier"] - eligible["ctr"]).clip(lower=0)
eligible["score"] = eligible["ctr_gap"] * eligible["total_impressions"]
eligible["reason_code"] = "ctr_below_tier_expectation"
eligible["action"] = "review_title_meta"

queue = eligible.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"{len(queue):,} pages scored and ranked. Wrote work/outputs/baseline_action_score.csv")
queue.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

43,619 pages scored and ranked. Wrote work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,avg_position,total_impressions,ctr,position_tier,expected_ctr_tier,ctr_gap,score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,7.346909,212404.0,0.000113,top_10,0.003208,0.003095,657.447318,ctr_below_tier_expectation,review_title_meta
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,4.545582,134984.0,0.000007,top_10,0.003208,0.003201,432.063807,ctr_below_tier_expectation,review_title_meta
2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,3.219473,143019.0,0.000301,top_10,0.003208,0.002908,415.842178,ctr_below_tier_expectation,review_title_meta
3,client_73cda7b4e4f265ea,content_fec55986a1868d62,9.385150,124075.0,0.000008,top_10,0.003208,0.003200,397.064895,ctr_below_tier_expectation,review_title_meta
4,client_62f4a7e64f5e0096,content_7c6373141eae744a,5.789019,132593.0,0.000626,top_10,0.003208,0.002582,342.392856,ctr_below_tier_expectation,review_title_meta
5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,9.536301,107584.0,0.000139,top_10,0.003208,0.003069,330.157475,ctr_below_tier_expectation,review_title_meta
6,client_62f4a7e64f5e0096,content_acbcc847f8996314,3.361195,170808.0,0.001534,top_10,0.003208,0.001674,285.996523,ctr_below_tier_expectation,review_title_meta
7,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,7.786219,89332.0,0.000045,top_10,0.003208,0.003163,282.600308,ctr_below_tier_expectation,review_title_meta
8,client_a80fca3f171ed1de,content_046fc480045b88f5,7.289152,83788.0,0.000072,top_10,0.003208,0.003137,262.813713,ctr_below_tier_expectation,review_title_meta
9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,4.450106,194337.0,0.001858,top_10,0.003208,0.001351,262.483680,ctr_below_tier_expectation,review_title_meta


## 3. Top-10 review

*For each of your top ten: one line each — the action, why it's there, and what would make it wrong.*


In [5]:
top10 = queue.head(10).copy()

for i, row in top10.iterrows():
    print(f"#{i+1}  content_hash_id={row['content_hash_id']}")
    print(f"    Action: {row['action']}  (reason: {row['reason_code']})")
    print(f"    Why: avg position {row['avg_position']:.1f} ({row['position_tier']}), "
          f"CTR {row['ctr']*100:.2f}% vs tier-expected {row['expected_ctr_tier']*100:.2f}% "
          f"on {row['total_impressions']:,.0f} impressions -> score {row['score']:.0f}")
    print(f"    Would be wrong if: this page's low CTR is explained by something my signals can't see — "
          f"e.g. a recent title change already in progress, a misleading-but-accurate snippet that filters "
          f"low-intent clicks on purpose, or a seasonal dip about to reverse on its own.")
    print()


#1  content_hash_id=content_44f34c0a90047651
    Action: review_title_meta  (reason: ctr_below_tier_expectation)
    Why: avg position 7.3 (top_10), CTR 0.01% vs tier-expected 0.32% on 212,404 impressions -> score 657
    Would be wrong if: this page's low CTR is explained by something my signals can't see — e.g. a recent title change already in progress, a misleading-but-accurate snippet that filters low-intent clicks on purpose, or a seasonal dip about to reverse on its own.

#2  content_hash_id=content_8e1334d6356668e3
    Action: review_title_meta  (reason: ctr_below_tier_expectation)
    Why: avg position 4.5 (top_10), CTR 0.00% vs tier-expected 0.32% on 134,984 impressions -> score 432
    Would be wrong if: this page's low CTR is explained by something my signals can't see — e.g. a recent title change already in progress, a misleading-but-accurate snippet that filters low-intent clicks on purpose, or a seasonal dip about to reverse on its own.

#3  content_hash_id=content_34a70f

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak-pick scan:** flagging any top-10 page whose `total_impressions` is close to my 500-impression floor — those pages have the least statistical stability, so a single unusual day could be doing most of the work in their score.


In [6]:
near_floor = top10[top10["total_impressions"] < 750]
if len(near_floor):
    print("Weak picks — near the volume floor, treat with less confidence:")
    print(near_floor[["content_hash_id", "total_impressions", "score"]])
else:
    print("No top-10 picks are near the volume floor — all comfortably above 750 impressions.")

print()
# Leakage check: only current-month, observed signals were used anywhere in this notebook.
used_columns = {"gsc_impressions", "gsc_clicks", "gsc_avg_position", "client_hash_id", "content_hash_id"}
print("Columns used in scoring:", used_columns)
print("No product flags (health_score, priority_score, action_type) were read from any table.")
print("No date outside", MONTH_START, "to", MONTH_END, "was queried for this rule.")


No top-10 picks are near the volume floor — all comfortably above 750 impressions.

Columns used in scoring: {'gsc_impressions', 'content_hash_id', 'gsc_avg_position', 'gsc_clicks', 'client_hash_id'}
No product flags (health_score, priority_score, action_type) were read from any table.
No date outside 2026-03-01 to 2026-04-01 was queried for this rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Both signal checks have a bucket table, a printed `n`, and a one-word verdict — at least one is flag-linked
- [ ] The rule has exactly one score, one reason code, one action label
- [ ] The notebook writes `work/outputs/baseline_action_score.csv` and it runs top to bottom with no errors
- [ ] Top 10 are reviewed, each with a "what would make it wrong" line
- [ ] No future-window or label-derived inputs anywhere in the rule
- [ ] No client names, URLs, or private queries anywhere
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
